# Model Optimization and Model Saving
## ML Modeling Challenge - Wizeline

Optimización de hiperparámetros con **RandomizedSearchCV** para el modelo ganador (**CatBoost**),
utilizando las features con importancia > 5 obtenidas en la sección 7 del Notebook 2.

**Métrica de selección**: menor **SMAPE de CV**

## 0. Imports y configuración

In [1]:
from pathlib import Path

import joblib
import numpy as np
import plotly.express as px
import polars as pl
import yaml
from catboost import CatBoostRegressor
from scipy.stats import loguniform, randint, uniform

from src.optimization_functions import get_optimization
from src.train_functions import (
    SEED,
    build_pipeline,
    fugacity_smape_score,
    mape_score,
    run_cross_validation,
    smape_score,
)

with open("config.yaml") as f:
    _cfg = yaml.safe_load(f)

CV_FOLDS: int = _cfg["model"]["cv_folds"]
MODEL_NAME: str = _cfg["model"]["name"]
ARTIFACT_PATH: str = _cfg["model"]["artifact_path"]
CV_METRICS_V2: dict = _cfg["model"]["cv_metrics_v2"]
N_ITER = 100  # Aumentar para búsqueda más exhaustiva (ej. 100)

print(f'Seed global:              {SEED}')
print(f'CV folds:                 {CV_FOLDS}')
print(f'Modelo base (v2):         {MODEL_NAME}')
print(f'Artifact path:            {ARTIFACT_PATH}')
print(f'Iteraciones de búsqueda:  {N_ITER}')
print(f'Métricas v2 (referencia): SMAPE={CV_METRICS_V2["smape_cv"]:.4f}%  R²={CV_METRICS_V2["r2_cv"]:.4f}  Fugacity={CV_METRICS_V2["fugacity_smape_cv"]:.4f}%')


Seed global:              5000
CV folds:                 5
Modelo base (v2):         CatBoost
Artifact path:            models/catboost_optimized.joblib
Iteraciones de búsqueda:  100
Métricas v2 (referencia): SMAPE=11.3860%  R²=0.8802  Fugacity=23.5000%


## 1. Carga de datos y selección de features

Se usa las features con **importancia > 5** obtenidas en la sección 7 del Notebook 2, cargadas desde `config.yaml` (`features.selected_v2`).

In [2]:
df_train = pl.read_csv('data/training_data.csv')

In [3]:
LIST_FEATURE_SELECTED_V2: list[str] = _cfg['features']['selected_v2']

X = df_train.select(LIST_FEATURE_SELECTED_V2).to_numpy()
y = df_train['target'].to_numpy()

print(f'Features seleccionadas ({len(LIST_FEATURE_SELECTED_V2)}): {LIST_FEATURE_SELECTED_V2}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')


Features seleccionadas (5): ['feature_2', 'feature_13', 'feature_9', 'feature_18', 'feature_11']
X shape: (800, 5)
y shape: (800,)


# 2. Espacio Hiperparametral

Se definen rangos para CatBoost incluyendo los valores **por default** de cada parámetro:

| Parámetro | Default | Rango de búsqueda | Distribución |
|---|---|---|---|
| `iterations` | 1000 | [100, 1000] | `randint` |
| `learning_rate` | 0.03 | [0.01, 0.30] | `loguniform` |
| `depth` | 6 | [4, 10] | `randint` |
| `l2_leaf_reg` | 3.0 | [1.0, 10.0] | `uniform` |
| `subsample` | 0.8 | [0.6, 1.0] | `uniform` |

In [4]:
param_space = {
    'model__iterations': randint(500, 1500),         # default: 1000
    'model__learning_rate': loguniform(0.029, 0.30), # default: 0.03
    'model__depth': randint(4, 11),                  # default: 6
    'model__l2_leaf_reg': uniform(1.0, 9.0),         # default: 3.0  → [1.0, 10.0]
    'model__subsample': uniform(0.6, 0.4),           # default: 0.8  → [0.6, 1.0]
}

## 3. RandomizedSearchCV

In [5]:
search = get_optimization(
    X=X,
    y=y,
    param_space=param_space,
    cv_folds=CV_FOLDS,
    n_iter=N_ITER,
    seed=SEED,
)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV 1/5; 1/100] START model__depth=7, model__iterations=950, model__l2_leaf_reg=4.2460278285460475, model__learning_rate=0.15876852781429254, model__subsample=0.6450671218880694
[CV 1/5; 1/100] END model__depth=7, model__iterations=950, model__l2_leaf_reg=4.2460278285460475, model__learning_rate=0.15876852781429254, model__subsample=0.6450671218880694; fugacity_smape: (test=-29.375) r2: (test=0.857) smape: (test=-12.186) total time=   0.9s
[CV 2/5; 1/100] START model__depth=7, model__iterations=950, model__l2_leaf_reg=4.2460278285460475, model__learning_rate=0.15876852781429254, model__subsample=0.6450671218880694
[CV 2/5; 1/100] END model__depth=7, model__iterations=950, model__l2_leaf_reg=4.2460278285460475, model__learning_rate=0.15876852781429254, model__subsample=0.6450671218880694; fugacity_smape: (test=-24.375) r2: (test=0.854) smape: (test=-11.326) total time=   0.8s
[CV 3/5; 1/100] START model__depth=7, model__iter

## 4. Mejor configuración de hiperparámetros

In [6]:
best_idx = int(np.argmax(search.cv_results_['mean_test_smape']))
best_params_raw: dict = search.cv_results_['params'][best_idx]
best_smape_cv = float(-search.cv_results_['mean_test_smape'][best_idx])
best_r2_cv = float(search.cv_results_['mean_test_r2'][best_idx])
best_fugacity_smape_cv = float(-search.cv_results_['mean_test_fugacity_smape'][best_idx])

print(f'Mejor trial         : #{best_idx}')
print(f'SMAPE CV            : {best_smape_cv:.4f}%')
print(f'R²    CV            : {best_r2_cv:.4f}')
print(f'Fugacity SMAPE CV   : {best_fugacity_smape_cv:.4f}%')
print('\nMejores hiperparámetros:')
for k, v in best_params_raw.items():
    clean_k = k.replace('model__', '')
    print(f'  {clean_k}: {v}')


Mejor trial         : #28
SMAPE CV            : 11.1985%
R²    CV            : 0.8852
Fugacity SMAPE CV   : 23.5000%

Mejores hiperparámetros:
  depth: 4
  iterations: 1498
  l2_leaf_reg: 9.262019577061952
  learning_rate: 0.032233613604091695
  subsample: 0.7340965149372803


## 5. Ajuste y Evaluación del modelo optimizado

Se entrena el modelo con la **mejor configuración** sobre todos los datos de entrenamiento
y se evalúa mediante Cross-Validation final.

In [7]:
# Extraer parámetros del estimador (quitar prefijo 'model__')
catboost_params: dict = {
    k.replace('model__', ''): v for k, v in best_params_raw.items()
}

best_pipeline = build_pipeline(
    CatBoostRegressor(**catboost_params, random_seed=SEED, verbose=0)
)
best_pipeline.fit(X, y)
print('Modelo ajustado sobre el conjunto de entrenamiento completo.')

Modelo ajustado sobre el conjunto de entrenamiento completo.


In [8]:
# run_cross_validation clona internamente antes de cada fold
metrics = run_cross_validation(best_pipeline, X, y, n_splits=CV_FOLDS, seed=SEED)

mape_cv = metrics['mape_cv']
smape_cv = metrics['smape_cv']
r2_cv = metrics['r2_cv']
fugacity_smape_cv = metrics['fugacity_smape_cv']

# Métricas sobre entrenamiento (informativo — sesgo optimista esperado)
y_pred_train = best_pipeline.predict(X)
mape_train = mape_score(y, y_pred_train)
smape_train = smape_score(y, y_pred_train)
r2_train = float(1 - np.sum((y - y_pred_train) ** 2) / np.sum((y - np.mean(y)) ** 2))
fugacity_train = fugacity_smape_score(y, y_pred_train)

print(f"{'Métrica':<22} {'Train':>10} {'CV':>10}")
print("-" * 44)
print(f"{'MAPE (%)':<22} {mape_train:>10.4f} {mape_cv:>10.4f}")
print(f"{'SMAPE (%)':<22} {smape_train:>10.4f} {smape_cv:>10.4f}")
print(f"{'R²':<22} {r2_train:>10.4f} {r2_cv:>10.4f}")
print(f"{'Fugacity SMAPE (%)':<22} {fugacity_train:>10.4f} {fugacity_smape_cv:>10.4f}")


Métrica                     Train         CV
--------------------------------------------
MAPE (%)                   7.3971    13.8620
SMAPE (%)                  6.5862    11.1985
R²                         0.9616     0.8852
Fugacity SMAPE (%)         7.5000    23.5000


## 6. Comparación v2 vs modelo optimizado

Se comparan las métricas de CV entre el **modelo base v2** (Notebook 2, sección 7) y el **modelo optimizado** (sección 5 de este notebook).

In [11]:
smape_v2 = CV_METRICS_V2["smape_cv"]
r2_v2 = CV_METRICS_V2["r2_cv"]
fugacity_v2 = CV_METRICS_V2["fugacity_smape_cv"]

print("--- Comparación v2 (base) vs Optimizado ---")

print(f"\nSMAPE v2        : {smape_v2:.4f}%")
print(f"SMAPE optimizado: {smape_cv:.4f}%")
pct_smape = (smape_cv - smape_v2) / smape_v2 * 100
print(f"Mejora          : {pct_smape:+.2f}% ({'mejor' if pct_smape < 0 else 'peor'} con optimización)")

print(f"\nR² v2           : {r2_v2:.4f}")
print(f"R² optimizado   : {r2_cv:.4f}")
pct_r2 = (r2_cv - r2_v2) / abs(r2_v2) * 100
print(f"Mejora          : {pct_r2:+.2f}% ({'mejor' if pct_r2 > 0 else 'peor'} con optimización)")

print(f"\nFugacity v2        : {fugacity_v2:.4f}%")
print(f"Fugacity optimizado: {fugacity_smape_cv:.4f}%")
pct_fugacity = (fugacity_smape_cv - fugacity_v2) / fugacity_v2 * 100
print(f"Mejora             : {pct_fugacity:+.2f}% (empate con optimización)")


--- Comparación v2 (base) vs Optimizado ---

SMAPE v2        : 11.3860%
SMAPE optimizado: 11.1985%
Mejora          : -1.65% (mejor con optimización)

R² v2           : 0.8802
R² optimizado   : 0.8852
Mejora          : +0.57% (mejor con optimización)

Fugacity v2        : 23.5000%
Fugacity optimizado: 23.5000%
Mejora             : +0.00% (empate con optimización)


## 7. Guardado del modelo

Se guarda el modelo optimizado usando **joblib** (más eficiente que pickle para objetos sklearn con arrays NumPy).

In [10]:
model_path = Path(ARTIFACT_PATH)
model_path.parent.mkdir(exist_ok=True)
joblib.dump(best_pipeline, model_path)

print(f'Modelo guardado en : {model_path}')
print(f'Tamaño             : {model_path.stat().st_size / 1024:.1f} KB')

# Guardar métricas CV del modelo optimizado en config.yaml
config_path = Path("config.yaml")
with open(config_path) as f:
    cfg = yaml.safe_load(f)

cfg["model"]["cv_metrics_optimized"] = {
    "mape_cv": round(float(mape_cv), 6),
    "smape_cv": round(float(smape_cv), 6),
    "r2_cv": round(float(r2_cv), 6),
    "fugacity_smape_cv": round(float(fugacity_smape_cv), 6),
}

with open(config_path, "w") as f:
    yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print(f'\nMétricas CV optimizado guardadas en {config_path}:')
for k, v in cfg["model"]["cv_metrics_optimized"].items():
    print(f'  {k}: {v}')


Modelo guardado en : models/catboost_optimized.joblib
Tamaño             : 485.3 KB

Métricas CV optimizado guardadas en config.yaml:
  mape_cv: 13.862046
  smape_cv: 11.198466
  r2_cv: 0.885167
  fugacity_smape_cv: 23.5
